# 시간대별 산불 발생 현황 분석 (1997 ~ 2025)
동국대 AI캠퍼스 · matplotlib 프로젝트 예제

- 데이터: KOSIS 시간대별 산불 발생 현황 (10년평균 · 금년, cp949)
- 흐름: 문제정의 → 데이터 준비(제공) → 시간대별 추세 → 금년 vs 평년 → 결론

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------
# 한글 폰트 설정 (OS 자동 감지)
# ---------------------------------------
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')     # 윈도우
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')        # 맥
else:
    plt.rc('font', family='NanumGothic')        # 리눅스 / 구글 코랩
plt.rc('axes', unicode_minus=False)
plt.rc('figure', figsize=(12, 5))

## 문제 정의
- 산불은 하루 중 **어느 시간대**에 가장 많이 발생하나?
- **2025년**은 **10년 평년**과 비교해 어땠나?

## 데이터 준비 (제공)
> **데이터 준비 코드는 미리 작성해 두었습니다(다단 헤더 처리는 pandas 심화 내용).**
>  아래 표(`cur`(금년)·`avg`(평년))는 **연도별로 정리된 쉬운 컬럼**이니, 우리는 **시각화에 집중**합니다.

In [ ]:
# ----------------------------------------
# [제공] 로드 + 정제 + 컬럼 정리 (인코딩 cp949)
# ----------------------------------------
data_file = 'data/산불데이터/시간대별_산불발생_현황_20260712203349.csv'
raw = pd.read_csv(data_file, header=[0, 1], encoding='cp949')
raw = raw.set_index(('시점', '시점'))
raw.index.name = '연도'
raw = raw.apply(lambda s: pd.to_numeric(
    s.astype(str).str.replace('-', '0').str.replace(',', ''), errors='coerce')).fillna(0)

cols = ['합계', '오전', '정오', '오후', '야간']
cur = raw['금년'].copy();     cur.columns = cols
avg = raw['10년 평균'].copy(); avg.columns = cols
cur.head()

## 시간대별 발생 추세 (금년 기준)

In [ ]:
times = ['오전', '정오', '오후', '야간']
time_colors = ['#A3D9FF', '#FFD966', '#FF9F68', '#3B3B98']
time_mean = cur[times].mean()
time_mean.round(1)

#### 시간대별 평균 발생 건수 (막대)
시간대는 **순서**가 있어 파이보다 막대가 적합하다. 최다 시간대만 강조.

In [ ]:
colors = ['#C0392B' if t == time_mean.idxmax() else '#BFC9CA' for t in times]
plt.figure(figsize=(8, 4))
plt.bar(times, time_mean.values, color=colors)
plt.title('시간대별 산불 평균 발생 건수 (최다 시간대 강조)')
plt.ylabel('건수')
plt.show()

#### 연도별 시간대 구성 (누적 막대)

In [ ]:
cur[times].plot(kind='bar', stacked=True, figsize=(14, 5), color=time_colors)
plt.title('연도별 시간대 구성 (발생 건수)')
plt.xlabel('연도'); plt.ylabel('건수'); plt.xticks(rotation=45)
plt.legend(title='시간대', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2025년 : 금년 vs 10년 평년

In [ ]:
cur_2025 = cur.loc[2025, times]
avg_2025 = avg.loc[2025, times]

x = np.arange(len(times))
plt.figure(figsize=(9, 5))
plt.bar(x - 0.2, cur_2025.values, width=0.4, label='2025 금년', color='#E67E22')
plt.bar(x + 0.2, avg_2025.values, width=0.4, label='10년 평균', color='#95A5A6')
plt.xticks(x, times)
plt.title('2025년 시간대별 발생 : 금년 vs 10년 평균')
plt.ylabel('건수'); plt.legend()
plt.show()

## 결론
- 산불은 **오후(14~18시)**에 가장 많이 발생하고 정오가 그 다음 → **한낮~오후**에 집중(기온↑·습도↓·야외활동↑).
- 야간·오전은 상대적으로 적다.
- 2025년도 같은 패턴(오후 최다)이며 평년과 분포 모양은 비슷 → 예방·감시 인력을 **오후에 집중** 배치하는 근거.